In [17]:
pip install "snowflake-snowpark-python[pandas]" snowflake-connector-python


Note: you may need to restart the kernel to use updated packages.


In [32]:
pip install snowflake-connector-python pandas streamlit

Note: you may need to restart the kernel to use updated packages.


In [33]:
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import DoubleType

In [34]:
connection_params={
"account":"GGGQHPR-MX77869",
   "user":"animelover",
    "password":"Animelover@02032006",
    "role":"ACCOUNTADMIN",
    "warehouse":"compute_wh",
    "database":"econ_agent_db",
    "schema":"analytics",

    
}
session=Session.builder.configs(connection_params).create()
print('connected to snowflake')

connected to snowflake


In [35]:
econ=session.table("snowflake_public_data_free.public_data_free.financial_economic_indicators_timeseries")

In [36]:
housing=session.table("snowflake_public_data_free.public_data_free.freddie_mac_housing_timeseries")

In [37]:
cpi=(
    econ.filter(F.col("GEO_ID")=="country/USA").filter(
        F.col("variable_name")
        =="CPI: All items, Monthly, 1982-84 Index Date (Seasonally adjusted)"
    
        ).select(
        F.col("DATE"),
        F.col("VALUE").cast(DoubleType()).alias("VALUE"),
        F.lit("CPI").alias("METRICS"),
        
        )

)
print("CPI rows:",cpi.count())

CPI rows: 949


In [38]:
unemployment=(
    econ.filter(F.col("GEO_ID")=="country/USA").filter(
        F.col("variable_name")
        =="Current Labor Force: Unemployment Rate - 20 yrs. & over, Monthly (Seasonally adjusted)"
    
        ).select(
        F.col("DATE"),
        F.col("VALUE").cast(DoubleType()).alias("VALUE"),
        F.lit("UNEMPLOYMENT_RATE").alias("METRICS"),
        
        )

)
print("unemployment rows:",unemployment.count())

unemployment rows: 937


In [39]:
mortage=(
    housing.filter(
        F.col("variable_name")
        =="30-Year Fixed Rate Mortgage Rate, National Average"
    
        ).select(
        F.col("DATE"),
        F.col("VALUE").cast(DoubleType()).alias("VALUE"),
        F.lit("MORTAGE_RATE_30").alias("METRICS"),
        
        )

)
print("mortage rows:",mortage.count())

mortage rows: 2870


In [40]:
combined=cpi.union_all(unemployment).union_all(mortage)
print('\n combined data sample')
combined.show(15)


 combined data sample
------------------------------------
|"DATE"      |"VALUE"  |"METRICS"  |
------------------------------------
|1995-10-31  |153.5    |CPI        |
|2002-12-31  |181.8    |CPI        |
|1968-05-31  |34.5     |CPI        |
|2000-09-30  |173.6    |CPI        |
|2003-07-31  |183.7    |CPI        |
|1964-08-31  |31.05    |CPI        |
|2025-11-30  |325.063  |CPI        |
|1987-02-28  |111.8    |CPI        |
|1983-05-31  |99.2     |CPI        |
|2010-04-30  |217.403  |CPI        |
|2024-02-29  |310.967  |CPI        |
|2024-11-30  |316.528  |CPI        |
|2022-09-30  |296.349  |CPI        |
|2022-05-31  |291.298  |CPI        |
|2024-08-31  |314.062  |CPI        |
------------------------------------



In [41]:
dashboard = (
    combined
    .group_by("DATE")
    .pivot(
        "METRICS",
        ["CPI", "Unemployment", "Mortgage"]
    )
    .agg(F.max("VALUE"))
    .sort(F.col("DATE").desc())
)

In [42]:
print("\nFinal Pivoted Dashboard")

dashboard.show()

print(f"Total rows: {dashboard.count()}")
print(f"Columns: {dashboard.columns}")


Final Pivoted Dashboard
----------------------------------------------------------
|"DATE"      |"'CPI'"  |"'Unemployment'"  |"'Mortgage'"  |
----------------------------------------------------------
|2026-03-26  |NULL     |NULL              |NULL          |
|2026-03-19  |NULL     |NULL              |NULL          |
|2026-03-12  |NULL     |NULL              |NULL          |
|2026-03-05  |NULL     |NULL              |NULL          |
|2026-02-28  |327.46   |NULL              |NULL          |
|2026-02-26  |NULL     |NULL              |NULL          |
|2026-02-19  |NULL     |NULL              |NULL          |
|2026-02-12  |NULL     |NULL              |NULL          |
|2026-02-05  |NULL     |NULL              |NULL          |
|2026-01-31  |326.588  |NULL              |NULL          |
----------------------------------------------------------

Total rows: 3714
Columns: ['DATE', '"\'CPI\'"', '"\'Unemployment\'"', '"\'Mortgage\'"']


In [43]:
dashboard = (
    combined
    .group_by("DATE")
    .pivot("METRICS", ["CPI", "Unemployment", "30 year mortgage"])
    .agg(F.max("VALUE"))
    .sort(F.col("DATE").desc())
)


In [44]:
print(combined.columns)

['DATE', 'VALUE', 'METRICS']


In [45]:
combined.show()

------------------------------------
|"DATE"      |"VALUE"  |"METRICS"  |
------------------------------------
|2024-02-29  |310.967  |CPI        |
|2024-11-30  |316.528  |CPI        |
|2022-09-30  |296.349  |CPI        |
|2022-05-31  |291.298  |CPI        |
|2024-08-31  |314.062  |CPI        |
|2023-04-30  |302.845  |CPI        |
|2025-02-28  |319.679  |CPI        |
|2022-04-30  |288.561  |CPI        |
|2026-01-31  |326.588  |CPI        |
|2025-06-30  |321.435  |CPI        |
------------------------------------

